# US 5.2 -- Diversity-Based Sample Selection

Uncertainty alone can lead to redundant selections -- the model might be uncertain
about many similar-looking images (e.g., all from the same part of the target
distribution).  Labeling five nearly identical images wastes the budget.

**Diversity selection** ensures the chosen samples cover as much of the target
distribution as possible.  We implement two strategies:

1. **Coreset selection** (farthest-first traversal): greedily pick the sample
   that is farthest from all already-selected samples in feature space.
2. **Clustering selection** (k-means): cluster the features into K groups and
   pick the sample closest to each cluster centre.

## What you will learn

1. How to extract features from the DANN encoder
2. Coreset selection: farthest-first traversal algorithm
3. Clustering selection: k-means in feature space
4. How to visualise selected samples in 2D (t-SNE)

## CLI equivalent

```bash
udm-epic5 select --strategy diversity --budget 50 --method coreset
udm-epic5 select --strategy diversity --budget 50 --method kmeans
```

## Prerequisites

- Python 3.12, PyTorch, scikit-learn
- Install: `uv pip install -e ".[epic5]"`

---
## 1. Extracting Features from the DANN Encoder

Both diversity strategies operate in **feature space**, not pixel space.  We use
the DANN shared encoder to extract a feature vector for each image.  The encoder
produces a bottleneck feature map (e.g., `[B, 768, 16, 16]` for ConvNeXt-Tiny),
which we global-average-pool to get a vector `[B, 768]`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from udm_epic4.models.dann import DANNModel
from udm_epic5.selection.diversity import coreset_selection, clustering_selection

In [ ]:
# Create a DANN model (in practice, load a trained checkpoint)
model = DANNModel(
    backbone="convnext_tiny",
    pretrained=False,
    in_chans=3,
    decoder_channels=[256, 128, 64, 32],
    domain_head_hidden=256,
)
model.eval()
print("Model ready for feature extraction.")

In [ ]:
# Simulate a pool of 200 target-domain images
n_pool = 200
dummy_pool = torch.randn(n_pool, 3, 512, 512)

# Extract features in batches
batch_size = 16
features_list = []

with torch.no_grad():
    for i in range(0, n_pool, batch_size):
        batch = dummy_pool[i:i+batch_size]
        feat_map = model.encode(batch)           # [B, 768, 16, 16]
        feat_vec = feat_map.mean(dim=[2, 3])     # [B, 768] global average pool
        features_list.append(feat_vec)

features = torch.cat(features_list, dim=0).numpy()  # [200, 768]
print(f"Feature matrix shape: {features.shape}")
print(f"Each image is represented by a {features.shape[1]}-dimensional vector.")

---
## 2. Coreset Selection: Farthest-First Traversal

The coreset algorithm is a greedy approach to select K samples that are
maximally spread out in feature space:

```
Algorithm: Farthest-First Traversal
1. Pick the first sample randomly (or use a seed).
2. For each remaining sample, compute its minimum distance to any
   already-selected sample.
3. Select the sample with the largest minimum distance.
4. Repeat until K samples are selected.
```

This guarantees that the selected samples form a **coreset** -- a compact
summary of the full dataset with good coverage.

In [ ]:
# Select 20 samples using coreset (farthest-first traversal)
budget = 20

coreset_indices = coreset_selection(
    features=features,
    budget=budget,
    seed=42,
)

print(f"Coreset selected {len(coreset_indices)} indices:")
print(f"  {coreset_indices}")
print()
print("These indices refer to positions in the target-domain pool.")
print("In practice, you map them back to image file paths.")

---
## 3. Clustering Selection: k-Means

An alternative is k-means clustering:

```
Algorithm: k-Means Selection
1. Run k-means on the feature matrix with K = budget.
2. For each cluster, find the sample closest to the cluster centre.
3. These K samples are the selected set.
```

k-Means tends to pick samples from dense regions (cluster centres), while
coreset tends to pick samples from sparse regions (outliers).  Both are valid
diversity strategies.

In [ ]:
# Select 20 samples using k-means clustering
kmeans_indices = clustering_selection(
    features=features,
    budget=budget,
    seed=42,
)

print(f"k-Means selected {len(kmeans_indices)} indices:")
print(f"  {kmeans_indices}")
print()

# Compare overlap between the two strategies
overlap = set(coreset_indices) & set(kmeans_indices)
print(f"Overlap between coreset and k-means: {len(overlap)} / {budget}")
print(f"The two strategies select different samples because they optimise")
print(f"different objectives (spread vs representativeness).")

---
## 4. Visualising Selected Samples in Feature Space

We use t-SNE to project the 768-dimensional features to 2D, then overlay
the selected samples.  This helps verify that the selection covers the
full distribution.

In [ ]:
# Compute t-SNE embedding
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embedding = tsne.fit_transform(features)  # [200, 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Coreset selection ---
ax = axes[0]
# All pool samples (gray)
ax.scatter(embedding[:, 0], embedding[:, 1],
           c='lightgray', s=20, alpha=0.6, label='Pool (unselected)')
# Selected samples (red)
ax.scatter(embedding[coreset_indices, 0], embedding[coreset_indices, 1],
           c='#F44336', s=80, edgecolors='black', linewidth=0.5,
           label=f'Coreset (n={budget})', zorder=5)
ax.set_title('Coreset Selection (Farthest-First)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(True, alpha=0.2)

# --- k-Means selection ---
ax = axes[1]
ax.scatter(embedding[:, 0], embedding[:, 1],
           c='lightgray', s=20, alpha=0.6, label='Pool (unselected)')
ax.scatter(embedding[kmeans_indices, 0], embedding[kmeans_indices, 1],
           c='#2196F3', s=80, edgecolors='black', linewidth=0.5,
           label=f'k-Means (n={budget})', zorder=5)
ax.set_title('Clustering Selection (k-Means)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(True, alpha=0.2)

fig.suptitle('Diversity Selection in Feature Space (t-SNE)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print("Left: coreset picks samples that are far apart (good spread).")
print("Right: k-means picks samples near cluster centres (representative).")
print("Both give good coverage of the target distribution.")

In [ ]:
# Quantitative check: average nearest-neighbour distance between selected samples
from scipy.spatial.distance import cdist

def mean_nn_distance(feat, indices):
    """Mean nearest-neighbour distance among selected samples."""
    selected = feat[indices]
    dists = cdist(selected, selected, metric='euclidean')
    np.fill_diagonal(dists, np.inf)
    return dists.min(axis=1).mean()

# Also compute for random selection as a baseline
np.random.seed(42)
random_indices = np.random.choice(n_pool, size=budget, replace=False)

print(f"Mean nearest-neighbour distance (higher = more diverse):")
print(f"  Random:   {mean_nn_distance(features, random_indices):.4f}")
print(f"  Coreset:  {mean_nn_distance(features, coreset_indices):.4f}")
print(f"  k-Means:  {mean_nn_distance(features, kmeans_indices):.4f}")
print()
print("Coreset typically achieves the highest spread.")

---
## 5. CLI Usage

```bash
# Coreset selection
udm-epic5 select \
    --strategy diversity \
    --method coreset \
    --budget 50 \
    --checkpoint checkpoints/dann_best.pt \
    --output results/selected_diversity.csv

# k-Means selection
udm-epic5 select \
    --strategy diversity \
    --method kmeans \
    --budget 50 \
    --checkpoint checkpoints/dann_best.pt \
    --output results/selected_diversity.csv
```

The output CSV has columns: `index`, `image_path`, `diversity_score`.

---
## Summary

- Diversity selection ensures the labeling budget covers the full target distribution.
- **Coreset** (farthest-first): maximises spread, good for outlier coverage.
- **k-Means**: picks representative samples near cluster centres.
- Both operate in the DANN encoder's feature space (768-d for ConvNeXt-Tiny).
- t-SNE visualisation confirms that selected samples are well spread.

**Next:** `epic5_03_combined.ipynb` -- combine uncertainty and diversity
into a single selection score.